# Functional demonstration: Shannon

Reduced model: 8 populations, 2 random seeds, and a limited parameter grid. These results are **not** the manuscript simulations. Run all cells in this notebook.


In [ ]:
# ============================================================
# Parameter scan:
#   sigma_base = 0.5, 0.6, ..., 1.5       (11 values)
#   L_var      = 0, 0.1, 0.2              (3 values)
#   max_inter  = 1, 5, 10                 (3 values)
#
# Each parameter combination:
#   10 random seeds
#   Treatment + dilution control
#
# Total:
#   11 × 3 × 3 = 99 scenarios
#   99 × 10 = 990 scenario-seed combinations
#   990 × 2 conditions = 1,980 time courses
# ============================================================

import numpy as np
from scipy.integrate import odeint


# ============================================================
# 1. Utility functions
# ============================================================

def generate_valid_random(N):
    """
    Generate N values from a standard normal distribution.

    The complete random vector is accepted only when:
        abs(mean) <= 0.1
        standard deviation <= 1.2

    This follows the random-number filtering used in the
    previous no-debris model.
    """
    while True:
        r = np.random.randn(N)

        if (
            abs(np.mean(r)) <= 0.1
            and np.std(r) <= 1.2
        ):
            return r


def param_generator_lv(
    N,
    connectedness,
    neg_frac,
    max_neg,
    max_pos
):
    """
    Generate a Lotka-Volterra interaction matrix.

    Parameters
    ----------
    N : int
        Number of species.

    connectedness : float
        Fraction of nonzero interspecies interactions.
        connectedness = 1 means fully connected.

    neg_frac : float
        Fraction of nonzero interactions that are negative.

    max_neg : float
        Maximum absolute strength of negative interactions.
        Negative interactions are sampled from:
            Uniform(-max_neg, 0)

    max_pos : float
        Maximum strength of positive interactions.
        Positive interactions are sampled from:
            Uniform(0, max_pos)

    Returns
    -------
    list
        [
            complete interaction matrix,
            negative interaction matrix,
            positive interaction matrix
        ]
    """

    total_off_diagonal = N * (N - 1)

    # Random ordering of all off-diagonal interaction positions
    inds_perm = np.random.permutation(total_off_diagonal)

    # Temporary matrix excluding the diagonal
    inter_init = np.zeros((N, N - 1))

    neg_count = int(
        np.ceil(
            total_off_diagonal
            * connectedness
            * neg_frac
        )
    )

    pos_count = int(
        np.floor(
            total_off_diagonal
            * connectedness
            * (1 - neg_frac)
        )
    )

    # Negative interactions
    if neg_count > 0 and max_neg > 0:
        inter_init.flat[
            inds_perm[:neg_count]
        ] = (
            -np.random.rand(neg_count)
            * max_neg
        )

    # Positive interactions
    if pos_count > 0 and max_pos > 0:
        start = neg_count
        end = neg_count + pos_count

        inter_init.flat[
            inds_perm[start:end]
        ] = (
            np.random.rand(pos_count)
            * max_pos
        )

    # Restore an N × N matrix with zero diagonal
    inter = np.zeros((N, N))

    for i in range(N):
        mask = np.ones(N, dtype=bool)
        mask[i] = False
        inter[i, mask] = inter_init[i, :]

    negative_inter = inter * (inter < 0)
    positive_inter = inter * (inter > 0)

    return [
        inter,
        negative_inter,
        positive_inter
    ]


# ============================================================
# 2. Mechanistic model
# ============================================================

def core_ode_lv_linparam(
        x, t, inter, gamma, beta,
        max_Gs, sigma, lambda_val
):
    # Effective growth rate
    G = (
        max_Gs * (1 - x)
        + inter @ x
    )

    # Only positive growth contributes to
    # growth-dependent lysis
    L = np.maximum(
    sigma * np.maximum(G, 0.0) + lambda_val,
    0.0
    )

    # Net population dynamics
    return x * (G - L)


def calculate_net_growth_rates(
    x,
    params,
    max_Gs,
    sigma,
    lambda_val
):
    """
    Return net population rate:

        N_i = G_i - L_i
    """

    G = (
        max_Gs * (1 - x)
        + params[0] @ x
    )

    L = np.maximum(
    sigma * np.maximum(G, 0.0) + lambda_val,
    0.0
    )

    return G - L


def calculate_effective_growth_rates(
    x,
    params,
    max_Gs
):
    """
    Return effective growth rate:

        G_i = max_G_i × (1 - x_i)
              + sum_j(inter_ij × x_j)
    """

    return (
        max_Gs * (1 - x)
        + params[0] @ x
    )

def calculate_lysis_rates(
        x, params, max_Gs,
        sigma, lambda_val
):
    G = (
        max_Gs * (1 - x)
        + params[0] @ x
    )

    L = np.maximum(
    sigma * np.maximum(G, 0.0) + lambda_val,
    0.0
    )

    return L


# ============================================================
# 3. Treatment simulation
# ============================================================

def simulate_treatment_timecourse(
    x0,
    tend,
    params,
    max_Gs,
    sigma,
    lambda_val,
    treatment_time=4.0
):
    """
    Treatment simulation.

    Phase 1: 0–4 h
        Growth only:
            sigma = 0
            lambda = 0

    Phase 2: 4–11 h
        Growth and lysis:
            sigma = species-specific treatment values
            lambda = species-specific treatment values

    The complete output contains 111 time points:
        0, 0.1, ..., 11 h
    """

    if treatment_time <= 0 or treatment_time >= tend:
        raise ValueError(
            "treatment_time must be between 0 and tend."
        )

    N = len(x0)

    # Complete output time grid
    t = np.linspace(
        0,
        tend,
        int(tend * 10) + 1
    )

    # Include exactly 4 h in phase 1
    t_phase1 = t[t <= treatment_time]

    # Times strictly after 4 h
    t_phase2_output = t[t > treatment_time]

    zero_vector = np.zeros(N)

    # --------------------------------------------------------
    # Phase 1: growth only
    # --------------------------------------------------------

    y1 = odeint(
        core_ode_lv_linparam,
        x0,
        t_phase1,
        args=(
            params[0],
            params[1],
            params[2],
            max_Gs,
            zero_vector,
            zero_vector
        ),
        atol=1e-9,
        rtol=1e-9
    )

    # --------------------------------------------------------
    # Phase 2: treatment begins at 4 h
    # --------------------------------------------------------

    # Add 4 h as the integration starting point
    t_phase2_ode = np.concatenate(
        ([treatment_time], t_phase2_output)
    )

    y2_with_initial = odeint(
        core_ode_lv_linparam,
        y1[-1],
        t_phase2_ode,
        args=(
            params[0],
            params[1],
            params[2],
            max_Gs,
            sigma,
            lambda_val
        ),
        atol=1e-9,
        rtol=1e-9
    )

    # Remove the duplicated 4 h initial row
    y2 = y2_with_initial[1:]

    # Complete population trajectory
    y = np.vstack([y1, y2])

    # Numerical protection against tiny negative abundances
    y = np.maximum(y, 0)

    # --------------------------------------------------------
    # Calculate rates
    # --------------------------------------------------------

    net_growth_rates = np.zeros_like(y)
    effective_growth_rates = np.zeros_like(y)
    lysis_rates = np.zeros_like(y)

    for i, current_time in enumerate(t):

        effective_growth_rates[i] = (
            calculate_effective_growth_rates(
                y[i],
                params,
                max_Gs
            )
        )

        if current_time < treatment_time:
            current_sigma = zero_vector
            current_lambda = zero_vector
        else:
            current_sigma = sigma
            current_lambda = lambda_val

        lysis_rates[i] = calculate_lysis_rates(
            y[i],
            params,
            max_Gs,
            current_sigma,
            current_lambda
        )

        net_growth_rates[i] = (
            effective_growth_rates[i]
            - lysis_rates[i]
        )

    return (
        t,
        y,
        net_growth_rates,
        effective_growth_rates,
        lysis_rates
    )


# ============================================================
# 4. Control simulation
# ============================================================

def simulate_control_timecourse(
    x0,
    tend,
    params,
    max_Gs,
    dilution_time=4.0,
    dilution_factor=25.0
):
    """
    Dilution control.

    Phase 1: 0–4 h
        Growth only.

    At exactly 4 h:
        Every species is diluted by the same factor.

    Phase 2: 4–11 h
        Growth only; no lysis.

    The stored abundance at 4 h is the post-dilution abundance.
    """

    if dilution_factor <= 0:
        raise ValueError(
            "dilution_factor must be greater than 0."
        )

    if dilution_time <= 0 or dilution_time >= tend:
        raise ValueError(
            "dilution_time must be between 0 and tend."
        )

    N = len(x0)

    t = np.linspace(
        0,
        tend,
        int(tend * 10) + 1
    )

    t_phase1 = t[t <= dilution_time]
    t_phase2_output = t[t > dilution_time]

    zero_vector = np.zeros(N)

    # --------------------------------------------------------
    # Phase 1: growth before dilution
    # --------------------------------------------------------

    y1 = odeint(
        core_ode_lv_linparam,
        x0,
        t_phase1,
        args=(
            params[0],
            params[1],
            params[2],
            max_Gs,
            zero_vector,
            zero_vector
        ),
        atol=1e-9,
        rtol=1e-9
    )

    # --------------------------------------------------------
    # Dilution at 4 h
    # --------------------------------------------------------

    y_after_dilution = (
        y1[-1] / dilution_factor
    )

    # Replace the 4 h pre-dilution abundance with the
    # post-dilution abundance
    y1[-1] = y_after_dilution

    # --------------------------------------------------------
    # Phase 2: growth after dilution
    # --------------------------------------------------------

    t_phase2_ode = np.concatenate(
        ([dilution_time], t_phase2_output)
    )

    y2_with_initial = odeint(
        core_ode_lv_linparam,
        y_after_dilution,
        t_phase2_ode,
        args=(
            params[0],
            params[1],
            params[2],
            max_Gs,
            zero_vector,
            zero_vector
        ),
        atol=1e-9,
        rtol=1e-9
    )

    y2 = y2_with_initial[1:]

    y = np.vstack([y1, y2])

    # Numerical protection
    y = np.maximum(y, 0)

    # --------------------------------------------------------
    # Calculate rates: no lysis at any time
    # --------------------------------------------------------

    net_growth_rates = np.zeros_like(y)
    effective_growth_rates = np.zeros_like(y)
    lysis_rates = np.zeros_like(y)

    for i in range(len(t)):

        effective_growth_rates[i] = (
            calculate_effective_growth_rates(
                y[i],
                params,
                max_Gs
            )
        )

        lysis_rates[i] = 0

        net_growth_rates[i] = (
            effective_growth_rates[i]
        )

    return (
        t,
        y,
        net_growth_rates,
        effective_growth_rates,
        lysis_rates
    )


# ============================================================
# 5. Fixed model parameters
# ============================================================

N = 8

# Equal initial abundance across 57 species
# Total initial abundance = 0.01
x0 = np.ones(N) / (100 * N)

tend = 11

# Same intrinsic growth-rate range as the previous
# no-debris simulation
max_Gs = np.linspace(
    0.7,
    1.5,
    N
)

# Baseline lysis rate
lambda_base = 0.1
lambda_val = (
    lambda_base * np.ones(N)
)

# Fully connected interaction network
connectedness = 1.0

# 50% negative and 50% positive interactions
neg_frac = 1

# Treatment/control timing
treatment_time = 4.0
dilution_time = 4.0
dilution_factor = 25.0


# ============================================================
# 6. Parameter scan
# ============================================================

# 0.5, 0.6, ..., 1.4
# np.round avoids floating-point labels such as 0.799999999
sigma_base_values = np.round(
    np.array([0.8, 1.2]),
    1
)

L_var_values = [0.0, 0.1]

max_inter_values = [1]

scenario_params = []

for sigma_base in sigma_base_values:
    for L_var in L_var_values:
        for max_inter in max_inter_values:

            scenario_params.append(
                {
                    "sigma_base": float(sigma_base),
                    "L_var": float(L_var),
                    "max_inter": int(max_inter)
                }
            )

n_scenarios = len(scenario_params)
n_seeds = 2

print(
    f"Number of sigma_base values: "
    f"{len(sigma_base_values)}"
)

print(
    f"Number of L_var values: "
    f"{len(L_var_values)}"
)

print(
    f"Number of max_inter values: "
    f"{len(max_inter_values)}"
)

print(
    f"Total scenarios: {n_scenarios}"
)

print(
    f"Scenario-seed combinations: "
    f"{n_scenarios * n_seeds}"
)

print(
    f"Treatment + control timecourses: "
    f"{n_scenarios * n_seeds * 2}"
)


# ============================================================
# 7. Initialize output arrays
# ============================================================

n_time = int(tend * 10) + 1
n_species = N

output_shape = (
    n_time,
    n_species,
    n_seeds,
    n_scenarios
)

# Treatment outputs
outputs = {
    "population_data":
        np.zeros(output_shape),

    "growth_rates":
        np.zeros(output_shape),

    "effective_growth_rates":
        np.zeros(output_shape),

    "lysis_rates":
        np.zeros(output_shape),

    "relative_abundance":
        np.zeros(output_shape)
}

# Control outputs
control_outputs = {
    "population_data":
        np.zeros(output_shape),

    "growth_rates":
        np.zeros(output_shape),

    "effective_growth_rates":
        np.zeros(output_shape),

    "lysis_rates":
        np.zeros(output_shape),

    "relative_abundance":
        np.zeros(output_shape)
}


# ============================================================
# 8. Run simulations
# ============================================================

for scenario_idx, scenario in enumerate(
    scenario_params
):

    sigma_base = scenario["sigma_base"]
    L_var = scenario["L_var"]
    max_inter = scenario["max_inter"]

    print(
        f"Scenario "
        f"{scenario_idx + 1}/{n_scenarios}"
        f" | sigma_base={sigma_base:.1f}"
        f" | L_var={L_var:.1f}"
        f" | max_inter={max_inter}"
    )

    for seed in range(1, n_seeds + 1):

        # Reproducible random values for this seed
        np.random.seed(seed)

        # ----------------------------------------------------
        # Species-specific lysis parameters
        # ----------------------------------------------------

        sigmas = (
            sigma_base
            + L_var
            * generate_valid_random(N)
        )

        lambdas = (
            lambda_base
            + L_var
            * generate_valid_random(N)
        )

        # This preserves the exact parameterization of the
        # previous no-debris model.
        #
        # If biologically required, the following two lines can
        # instead be activated to prevent negative values:
        #
        # sigmas = np.clip(sigmas, 0, None)
        # lambdas = np.clip(lambdas, 0, None)

        # ----------------------------------------------------
        # Interaction matrix
        # ----------------------------------------------------

        # max_Gs[-1] = 1.5
        current_max_inter = (
            max_Gs[-1] * max_inter
            if max_inter > 0
            else 0
        )

        params = param_generator_lv(
            N=N,
            connectedness=connectedness,
            neg_frac=neg_frac,
            max_neg=current_max_inter,
            max_pos=current_max_inter * 0.5
        )

        # Therefore:
        #
        # Negative interactions:
        #     U(-1.5 × max_inter, 0)
        #
        # Positive interactions:
        #     U(0, 0.75 × max_inter)

        # ----------------------------------------------------
        # Treatment
        # ----------------------------------------------------

        (
            t,
            y,
            net_rates,
            effective_rates,
            lysis_rates
        ) = simulate_treatment_timecourse(
            x0=x0,
            tend=tend,
            params=params,
            max_Gs=max_Gs,
            sigma=sigmas,
            lambda_val=lambdas,
            treatment_time=treatment_time
        )

        outputs[
            "population_data"
        ][:, :, seed - 1, scenario_idx] = y

        outputs[
            "growth_rates"
        ][:, :, seed - 1, scenario_idx] = (
            net_rates
        )

        outputs[
            "effective_growth_rates"
        ][:, :, seed - 1, scenario_idx] = (
            effective_rates
        )

        outputs[
            "lysis_rates"
        ][:, :, seed - 1, scenario_idx] = (
            lysis_rates
        )

        treatment_total = y.sum(
            axis=1,
            keepdims=True
        )

        outputs[
            "relative_abundance"
        ][:, :, seed - 1, scenario_idx] = (
            np.divide(
                y,
                treatment_total,
                out=np.zeros_like(y),
                where=treatment_total > 0
            )
        )

        # ----------------------------------------------------
        # Dilution control
        # ----------------------------------------------------

        (
            t_ctrl,
            y_ctrl,
            net_rates_ctrl,
            effective_rates_ctrl,
            lysis_rates_ctrl
        ) = simulate_control_timecourse(
            x0=x0,
            tend=tend,
            params=params,
            max_Gs=max_Gs,
            dilution_time=dilution_time,
            dilution_factor=dilution_factor
        )

        # Confirm identical output time grids
        if not np.allclose(t, t_ctrl):
            raise RuntimeError(
                "Treatment and control time grids differ."
            )

        control_outputs[
            "population_data"
        ][:, :, seed - 1, scenario_idx] = (
            y_ctrl
        )

        control_outputs[
            "growth_rates"
        ][:, :, seed - 1, scenario_idx] = (
            net_rates_ctrl
        )

        control_outputs[
            "effective_growth_rates"
        ][:, :, seed - 1, scenario_idx] = (
            effective_rates_ctrl
        )

        control_outputs[
            "lysis_rates"
        ][:, :, seed - 1, scenario_idx] = (
            lysis_rates_ctrl
        )

        control_total = y_ctrl.sum(
            axis=1,
            keepdims=True
        )

        control_outputs[
            "relative_abundance"
        ][:, :, seed - 1, scenario_idx] = (
            np.divide(
                y_ctrl,
                control_total,
                out=np.zeros_like(y_ctrl),
                where=control_total > 0
            )
        )


# ============================================================
# 9. Convert scenario metadata to simple arrays
# ============================================================

scenario_sigma_base = np.array(
    [
        scenario["sigma_base"]
        for scenario in scenario_params
    ],
    dtype=float
)

scenario_L_var = np.array(
    [
        scenario["L_var"]
        for scenario in scenario_params
    ],
    dtype=float
)

scenario_max_inter = np.array(
    [
        scenario["max_inter"]
        for scenario in scenario_params
    ],
    dtype=int
)


# ============================================================
# 10. Save all results
# ============================================================

output_filename = (
    "demo_shannon_results.npz"
)

np.savez_compressed(
    output_filename,

    # General metadata
    time_points=t,
    seed_values=np.arange(1, n_seeds + 1),
    species_index=np.arange(1, N + 1),

    # Scenario metadata
    sigma_base_values=sigma_base_values,
    L_var_values=np.array(L_var_values),
    max_inter_values=np.array(max_inter_values),

    scenario_sigma_base=scenario_sigma_base,
    scenario_L_var=scenario_L_var,
    scenario_max_inter=scenario_max_inter,

    # Fixed parameters
    initial_abundance=x0,
    intrinsic_growth_rates=max_Gs,
    lambda_base=lambda_base,
    connectedness=connectedness,
    negative_interaction_fraction=neg_frac,
    treatment_time=treatment_time,
    dilution_time=dilution_time,
    dilution_factor=dilution_factor,

    # Treatment outputs
    population_data=
        outputs["population_data"],

    growth_rates=
        outputs["growth_rates"],

    effective_growth_rates=
        outputs["effective_growth_rates"],

    lysis_rates=
        outputs["lysis_rates"],

    relative_abundance=
        outputs["relative_abundance"],

    # Control outputs
    control_population_data=
        control_outputs["population_data"],

    control_growth_rates=
        control_outputs["growth_rates"],

    control_effective_growth_rates=
        control_outputs[
            "effective_growth_rates"
        ],

    control_lysis_rates=
        control_outputs["lysis_rates"],

    control_relative_abundance=
        control_outputs[
            "relative_abundance"
        ]
)

print("\nFinished.")
print(f"Results saved to: {output_filename}")
print(f"Treatment array shape: {outputs['population_data'].shape}")
print(
    f"Control array shape: "
    f"{control_outputs['population_data'].shape}"
)